In [55]:
from sklearn.datasets import make_classification
import pandas as pd
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
import numpy as np
from sklearn.model_selection import TimeSeriesSplit, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
import xgboost as xgb

np.random.seed(42)

X_raw, y = make_classification(
    n_samples=50_000, n_features=15, n_informative=6,
    weights=[0.98, 0.02], random_state=42
)
df = pd.DataFrame(X_raw, columns=[f'feat_{i}' for i in range(15)])
df['merchant_category'] = np.random.choice(['retail','travel','food','online'], len(df))
df['txn_timestamp'] = pd.date_range('2024-01-01', periods=len(df), freq='1min')
df['is_fraud'] = y
df['fraud_confirmed_flag'] = (y == 1).astype(float) + np.random.normal(0, 0.05, len(y))

# Bug 1 fix: assign back
df = df.drop(['fraud_confirmed_flag'], axis=1)

# Bug 2 fix: assign back
df = df.sort_values('txn_timestamp').reset_index(drop=True)

# Bug 3 fix: drop txn_timestamp from X
X = df.drop(['is_fraud', 'txn_timestamp'], axis=1)
y = df['is_fraud']

num_features = [f'feat_{i}' for i in range(15)]
cat_features = ['merchant_category']

num_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
])

cat_pipeline = Pipeline([
    ('ordinal', OrdinalEncoder(categories=[['food','online','retail','travel']],
                               handle_unknown='use_encoded_value',
                               unknown_value=-1)),
])

pre = ColumnTransformer(transformers=[
    ('num', num_pipe, num_features),
    ('cat', cat_pipeline, cat_features),
])

# Bug 4 fix: removed redundant SimpleImputer
pipe = Pipeline([
    ('pre', pre),
    ('clf', xgb.XGBClassifier(scale_pos_weight=49, max_depth=4, eval_metric='aucpr')),
])

scores = cross_val_score(
    pipe, X, y,
    cv=TimeSeriesSplit(n_splits=5, gap=100),
    scoring='average_precision'
)

print("PR-AUC per fold:", scores)
print(f"Mean PR-AUC: {scores.mean():.4f} ± {scores.std():.4f}")

PR-AUC per fold: [0.37865326 0.46568781 0.50666373 0.52034086 0.59831908]
Mean PR-AUC: 0.4939 ± 0.0719


In [56]:
from sklearn.datasets import make_classification
import pandas as pd
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
import numpy as np
from sklearn.model_selection import TimeSeriesSplit, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
import xgboost as xgb

np.random.seed(42)

X_raw, y_raw = make_classification(
    n_samples=50_000, n_features=15, n_informative=6,
    weights=[0.98, 0.02], random_state=42
)
df = pd.DataFrame(X_raw, columns=[f'feat_{i}' for i in range(15)])
df['merchant_category'] = np.random.choice(['retail','travel','food','online'], len(df))
df['txn_timestamp'] = pd.date_range('2024-01-01', periods=len(df), freq='1min')
df['is_fraud'] = y_raw
df['fraud_confirmed_flag'] = (y_raw == 1).astype(float) + np.random.normal(0, 0.05, len(y_raw))

df = df.sort_values('txn_timestamp').reset_index(drop=True)

num_features = [f'feat_{i}' for i in range(15)]
cat_features  = ['merchant_category']

def build_pipe(extra_num_features=None):
    num_feats = num_features + (extra_num_features or [])

    num_pipe = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler',  StandardScaler()),
    ])
    cat_pipe = Pipeline([
        ('ordinal', OrdinalEncoder(
            categories=[['food','online','retail','travel']],
            handle_unknown='use_encoded_value',
            unknown_value=-1
        )),
    ])
    pre = ColumnTransformer(transformers=[
        ('num', num_pipe, num_feats),
        ('cat', cat_pipe, cat_features),
    ])
    return Pipeline([
        ('pre', pre),
        ('clf', xgb.XGBClassifier(
            scale_pos_weight=49, max_depth=4,
            eval_metric='aucpr', random_state=42
        )),
    ])

cv = TimeSeriesSplit(n_splits=5, gap=100)

# --- RUN 1: Clean (leaky feature dropped) ---
X_clean = df.drop(['is_fraud', 'txn_timestamp', 'fraud_confirmed_flag'], axis=1)
y = df['is_fraud']

scores_clean = cross_val_score(build_pipe(), X_clean, y, cv=cv, scoring='average_precision')

# --- RUN 2: Leaked (leaky feature kept) ---
X_leaky = df.drop(['is_fraud', 'txn_timestamp'], axis=1)

scores_leaky = cross_val_score(
    build_pipe(extra_num_features=['fraud_confirmed_flag']),
    X_leaky, y, cv=cv, scoring='average_precision'
)

# --- Report ---
print("=" * 45)
print("  LEAKAGE VERIFICATION REPORT")
print("=" * 45)
print(f"\n CLEAN  — PR-AUC per fold: {np.round(scores_clean, 4)}")
print(f"          Mean: {scores_clean.mean():.4f} ± {scores_clean.std():.4f}")
print(f"\n LEAKY  — PR-AUC per fold: {np.round(scores_leaky, 4)}")
print(f"          Mean: {scores_leaky.mean():.4f} ± {scores_leaky.std():.4f}")
print(f"\n GAP    — {scores_leaky.mean() - scores_clean.mean():.4f}")
print("\n VERDICT:", end=" ")
if scores_leaky.mean() - scores_clean.mean() > 0.30:
    print("LEAKAGE CONFIRMED. Pipeline correctly isolates it when dropped.")
else:
    print("Gap too small — check if leaky feature was actually removed.")

  LEAKAGE VERIFICATION REPORT

 CLEAN  — PR-AUC per fold: [0.3787 0.4657 0.5067 0.5203 0.5983]
          Mean: 0.4939 ± 0.0719

 LEAKY  — PR-AUC per fold: [0.9999 0.9984 0.9989 0.9991 0.9979]
          Mean: 0.9989 ± 0.0007

 GAP    — 0.5049

 VERDICT: LEAKAGE CONFIRMED. Pipeline correctly isolates it when dropped.


In [ ]:
split_idx=int(len(df)*0.80)

train_df=df.iloc[:split_idx].copy()
test_df=df.iloc[split_idx:].copy()